In [ ]:
# Install necessary libraries
!pip install torch torchvision opencv-python numpy ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.2/949.2 kB 55.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [ ]:
!pip install timm

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from torchvision import transforms
from PIL import Image

# Load YOLO Model (Ensure you have your trained model file)
yolo_model = YOLO("/content/best (1).pt")  # Change path to your trained YOLOv8 model

# Load MiDaS Model for Depth Estimation
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.eval()

# MiDaS Image Preprocessing
transform = transforms.Compose([
    transforms.Resize((384, 384)),  # Resize to model input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Loading weights:  None


Using cache found in /root/.cache/torch/hub/rwightman_gen-efficientnet-pytorch_master


In [ ]:
def detect_nail_and_depth(image_path, reference_width_mm=10):
    """
    Detects nails using YOLO, estimates depth with MiDaS, and calculates real-world size.

    :param image_path: Path to input image.
    :param reference_width_mm: Real-world width of a reference object in mm.
    :return: Bounding box, estimated depth, and depth map.
    """

    # Load Image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Step 1: Run YOLO for Nail Detection
    results = yolo_model(img_rgb)
    detections = results[0].boxes.xyxy.cpu().numpy()  # Extract (x_min, y_min, x_max, y_max)

    if len(detections) == 0:
        print("❌ No nails detected!")
        return None, None, None

    # Get first detected nail bounding box
    x_min, y_min, x_max, y_max = map(int, detections[0][:4])

    # Step 2: Run MiDaS for Depth Estimation
    img_pil = Image.fromarray(img_rgb)
    img_transformed = transform(img_pil).unsqueeze(0)  # Shape: [1, 3, 384, 384]

    with torch.no_grad():
        depth_map = midas(img_transformed)  # MiDaS prediction
    depth_map = depth_map.squeeze().cpu().numpy()

    # Normalize depth map for visualization
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min()) * 255
    depth_map = depth_map.astype(np.uint8)

    # Step 3: Extract Depth of the Nail
    nail_depth_values = depth_map[y_min:y_max, x_min:x_max]
    avg_depth = np.median(nail_depth_values)

    # Step 4: Estimate Nail Length & Width in Real-World Units
    focal_length = 500  # Adjust based on your camera
    pixel_width = x_max - x_min  # Width in pixels

    real_width_mm = (reference_width_mm * focal_length) / avg_depth  # Convert to mm

    print(f"📏 Estimated Nail Width: {real_width_mm:.2f} mm")

    return (x_min, y_min, x_max, y_max), avg_depth, depth_map


In [ ]:
image_path = "/content/WhatsApp Image 2024-09-01 at 8.35.02 PM.jpeg"  # Upload an image to Colab first

nail_bbox, depth, depth_map = detect_nail_and_depth(image_path)

if depth_map is not None:
    plt.imshow(depth_map, cmap="plasma")
    plt.colorbar()
    plt.show()



0: 640x384 6 Nails, 456.8ms
Speed: 4.5ms preprocess, 456.8ms inference, 16.1ms postprocess per image at shape (1, 3, 640, 384)
📏 Estimated Nail Width: nan mm


In [ ]:
image_path = "/content/WhatsApp Image 2024-09-01 at 8.35.02 PM.jpeg"

# Read the image
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Run YOLO on the image
results = yolo_model(img_rgb)

# Check the output format
print("YOLO Output:", results)



0: 640x384 6 Nails, 477.0ms
Speed: 5.1ms preprocess, 477.0ms inference, 25.9ms postprocess per image at shape (1, 3, 640, 384)
YOLO Output: [ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: ultralytics.engine.results.Masks object
names: {0: 'Nail'}
obb: None
orig_img: array([[[ 82,  85,  58],
        [ 82,  85,  58],
        [ 83,  86,  59],
        ...,
        [  7,   0,   0],
        [ 26,  16,  15],
        [ 67,  57,  56]],

       [[ 85,  88,  61],
        [ 85,  88,  61],
        [ 85,  88,  61],
        ...,
        [  7,   0,   0],
        [ 10,   0,   0],
        [ 19,   9,   8]],

       [[ 88,  90,  66],
        [ 88,  90,  66],
        [ 89,  89,  65],
        ...,
        [ 15,   5,   4],
        [ 10,   0,   0],
        [  7,   0,   0]],

       ...,

       [[119,  99,  48],
        [122, 104,  54],
        [121, 102,  59],
        ...,
        [151, 136, 115],
        [153, 138, 117],
   

In [ ]:
# Extract bounding box coordinates correctly
if hasattr(results, 'boxes'):
    detections = results.boxes.xyxy.cpu().numpy()  # Convert to NumPy array

    if len(detections) > 0:
        for i, (x1, y1, x2, y2) in enumerate(detections[:, :4]):
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            print(f"Detection {i+1}: Bounding Box ({x1}, {y1}) to ({x2}, {y2})")
    else:
        print("No nails detected!")

else:
    print("Error: No 'boxes' attribute found in YOLO results!")


Error: No 'boxes' attribute found in YOLO results!


In [ ]:
print(yolo_model)


YOLO(
  (model): SegmentationModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (m): ModuleList(
          (0): Bottleneck(
            (cv1): Conv(
              (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)
            )
            (cv2): Conv(
              (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)


In [ ]:
print(results)


[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: ultralytics.engine.results.Masks object
names: {0: 'Nail'}
obb: None
orig_img: array([[[ 82,  85,  58],
        [ 82,  85,  58],
        [ 83,  86,  59],
        ...,
        [  7,   0,   0],
        [ 26,  16,  15],
        [ 67,  57,  56]],

       [[ 85,  88,  61],
        [ 85,  88,  61],
        [ 85,  88,  61],
        ...,
        [  7,   0,   0],
        [ 10,   0,   0],
        [ 19,   9,   8]],

       [[ 88,  90,  66],
        [ 88,  90,  66],
        [ 89,  89,  65],
        ...,
        [ 15,   5,   4],
        [ 10,   0,   0],
        [  7,   0,   0]],

       ...,

       [[119,  99,  48],
        [122, 104,  54],
        [121, 102,  59],
        ...,
        [151, 136, 115],
        [153, 138, 117],
        [155, 140, 119]],

       [[117,  97,  47],
        [120, 102,  54],
        [114,  96,  50],
        ...,
        [149, 134, 113],
    

In [ ]:
print(dir(results))  # Check what attributes exist in 'results'


['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__imul__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__rmul__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']


In [ ]:
if isinstance(results, list):
    results = results[0]  # Get the first result

if hasattr(results, "boxes") and results.boxes is not None:
    detections = results.boxes.xyxy.cpu().numpy()  # Extract bounding boxes
    print(f"Detections found: {len(detections)}")

    for i, (x1, y1, x2, y2) in enumerate(detections[:, :4]):
        print(f"Nail {i+1}: Bounding Box ({int(x1)}, {int(y1)}) to ({int(x2)}, {int(y2)})")
else:
    print("No nails detected!")


Detections found: 6
Nail 1: Bounding Box (948, 1291) to (1012, 1362)
Nail 2: Bounding Box (365, 887) to (413, 943)
Nail 3: Bounding Box (762, 805) to (814, 863)
Nail 4: Bounding Box (601, 740) to (646, 793)
Nail 5: Bounding Box (500, 762) to (558, 820)
Nail 6: Bounding Box (368, 879) to (417, 939)


In [ ]:
# Check if depth_map exists
if 'depth_map' in locals():
    print("Depth map shape:", depth_map.shape)
else:
    print("❌ No depth map found!")


Depth map shape: (384, 384)


In [ ]:
print(depth_map[:5, :5])  # Show top-left corner


[[3 4 4 4 4]
 [4 4 4 4 4]
 [4 4 4 4 4]
 [5 5 5 5 5]
 [5 5 5 5 5]]


In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torchvision.transforms as transforms
from torchvision import models

# Load YOLO model
yolo_model = YOLO("/content/best (1).pt")  # Replace with your trained YOLO model

# Load MiDaS depth estimation model
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")  # Use "MiDaS_large" for better accuracy
midas.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)

# Define MiDaS image transformation
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((384, 384)),  # MiDaS expects 384x384 or 256x256 images
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])



Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Loading weights:  None


Using cache found in /root/.cache/torch/hub/rwightman_gen-efficientnet-pytorch_master


In [ ]:
# Load image
image_path = "/content/test.jfif"  # Replace with your image
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Run YOLO detection
yolo_results = yolo_model(image)

# Extract bounding boxes
if yolo_results[0].boxes is not None:
    detections = yolo_results[0].boxes.xyxy.cpu().numpy()
    print(f"Detections found: {len(detections)}")

    # Convert image to MiDaS input format
    image_resized = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_resized = transform(image_resized).unsqueeze(0).to(device)

    # Run MiDaS depth estimation
    with torch.no_grad():
        depth_map = midas(image_resized)

    depth_map = depth_map.squeeze().cpu().numpy()
    depth_map = cv2.resize(depth_map, (image.shape[1], image.shape[0]))

    # Normalize depth values
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())

    # Process detections
    for i, (x1, y1, x2, y2) in enumerate(detections[:, :4]):
        width_px = x2 - x1
        length_px = y2 - y1

        # Get average depth for the nail region
        nail_depth = np.mean(depth_map[int(y1):int(y2), int(x1):int(x2)])

        # Convert to real-world units (Assume 1 depth unit = 10 mm scale factor)
        scale_factor = 10 * nail_depth  # Adjust based on reference object
        width_mm = width_px / scale_factor
        length_mm = length_px / scale_factor

        # Draw bounding box
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(image, f"Nail {i+1}: {width_mm:.2f}mm x {length_mm:.2f}mm",
                    (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        print(f"Nail {i+1}: Width = {width_mm:.2f} mm, Length = {length_mm:.2f} mm")

# Show result
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Nail Measurement with Depth Estimation")
plt.show()


0: 640x384 4 Nails, 638.6ms
Speed: 3.7ms preprocess, 638.6ms inference, 12.2ms postprocess per image at shape (1, 3, 640, 384)
Detections found: 4
Nail 1: Width = 11.22 mm, Length = 15.89 mm
Nail 2: Width = 12.88 mm, Length = 19.11 mm
Nail 3: Width = 17.39 mm, Length = 27.56 mm
Nail 4: Width = 7.20 mm, Length = 11.64 mm


In [ ]:
# Load image
image_path = "/content/test2.jfif"  # Replace with your image
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Run YOLO detection
yolo_results = yolo_model(image)

# Extract bounding boxes
if yolo_results[0].boxes is not None:
    detections = yolo_results[0].boxes.xyxy.cpu().numpy()
    print(f"Detections found: {len(detections)}")

    # Convert image to MiDaS input format
    image_resized = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_resized = transform(image_resized).unsqueeze(0).to(device)

    # Run MiDaS depth estimation
    with torch.no_grad():
        depth_map = midas(image_resized)

    depth_map = depth_map.squeeze().cpu().numpy()
    depth_map = cv2.resize(depth_map, (image.shape[1], image.shape[0]))

    # Normalize depth values
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())

    # Process detections
    for i, (x1, y1, x2, y2) in enumerate(detections[:, :4]):
        width_px = x2 - x1
        length_px = y2 - y1

        # Get average depth for the nail region
        nail_depth = np.mean(depth_map[int(y1):int(y2), int(x1):int(x2)])

        # Convert to real-world units (Assume 1 depth unit = 10 mm scale factor)
        scale_factor = 10 * nail_depth  # Adjust based on reference object
        width_mm = width_px / scale_factor
        length_mm = length_px / scale_factor

        # Draw bounding box
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(image, f"Nail {i+1}: {width_mm:.2f}mm x {length_mm:.2f}mm",
                    (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        print(f"Nail {i+1}: Width = {width_mm:.2f} mm, Length = {length_mm:.2f} mm")

# Show result
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Nail Measurement with Depth Estimation")
plt.show()


0: 640x288 4 Nails, 1126.4ms
Speed: 3.8ms preprocess, 1126.4ms inference, 24.7ms postprocess per image at shape (1, 3, 640, 288)
Detections found: 4
Nail 1: Width = 15.53 mm, Length = 22.34 mm
Nail 2: Width = 12.91 mm, Length = 18.96 mm
Nail 3: Width = 21.25 mm, Length = 31.80 mm
Nail 4: Width = 10.71 mm, Length = 16.99 mm


In [ ]:
import torch
import cv2
import numpy as np
from ultralytics import YOLO
from torchvision.transforms import Compose, Normalize, ToTensor
from PIL import Image
import matplotlib.pyplot as plt

# Load YOLO Nail Detection Model
yolo_model = YOLO("/content/best (1).pt")  # Replace with your trained YOLO model

# Load MiDaS Depth Estimation Model
midas = torch.hub.load("intel-isl/MiDaS", "DPT_Large")  # Use DPT_Large for accuracy
midas.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)

# MiDaS preprocessing transforms
midas_transform = transforms.Compose([
    transforms.Resize((384, 384)),  # Ensure correct input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


In [ ]:
def detect_nails(image):
    """Detect nails using YOLO model."""
    results = yolo_model(image)
    return results[0].boxes.xyxy.cpu().numpy()


def estimate_depth(image):
    """Estimate depth using MiDaS model with proper preprocessing."""
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)  # Convert to PIL Image for torchvision transforms

    input_tensor = midas_transform(img_pil).unsqueeze(0).to(device)  # Add batch dimension

    with torch.no_grad():
        depth_map = midas(input_tensor)

    depth_map = depth_map.squeeze().cpu().numpy()
    depth_map = cv2.resize(depth_map, (image.shape[1], image.shape[0]))  # Resize back to original size
    return depth_map


def get_real_world_size(bbox, depth_map, reference_mm=85.6):
    """Compute real-world width and length using depth estimation."""
    x1, y1, x2, y2 = map(int, bbox)
    depth = np.median(depth_map[y1:y2, x1:x2])  # Get depth from the detected region

    # Reference object width (Credit Card ~ 85.6mm)
    ref_width_px = 85.6
    pixel_to_mm = reference_mm / ref_width_px

    width = (x2 - x1) * pixel_to_mm * depth
    length = (y2 - y1) * pixel_to_mm * depth

    return round(width, 2), round(length, 2)


def main(image_path):
    """Run nail detection and measurement pipeline."""
    image = cv2.imread(image_path)

    # Detect nails using YOLO
    boxes = detect_nails(image)

    # Estimate depth using MiDaS
    depth_map = estimate_depth(image)

    print(f"Detections found: {len(boxes)}")
    for i, bbox in enumerate(boxes):
        width, length = get_real_world_size(bbox, depth_map)
        print(f"Nail {i+1}: Width = {width} mm, Length = {length} mm")

        x1, y1, x2, y2 = map(int, bbox)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, f"{width}mm x {length}mm", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


# Run the script
image_path = "/content/ok.jfif"  # Replace with your image path
main(image_path)


0: 640x480 4 Nails, 1421.6ms
Speed: 5.9ms preprocess, 1421.6ms inference, 13.9ms postprocess per image at shape (1, 3, 640, 480)
Detections found: 4
Nail 1: Width = 2362.56 mm, Length = 2759.33 mm
Nail 2: Width = 2761.92 mm, Length = 3131.47 mm
Nail 3: Width = 2236.04 mm, Length = 2552.81 mm
Nail 4: Width = 2299.78 mm, Length = 2930.69 mm


In [ ]:
# Run the script
image_path = "/content/oka.jfif"  # Replace with your image path
main(image_path)


0: 640x480 4 Nails, 791.1ms
Speed: 5.5ms preprocess, 791.1ms inference, 13.4ms postprocess per image at shape (1, 3, 640, 480)
Detections found: 4
Nail 1: Width = 2166.74 mm, Length = 2530.9 mm
Nail 2: Width = 2321.25 mm, Length = 2575.14 mm
Nail 3: Width = 2348.82 mm, Length = 2690.47 mm
Nail 4: Width = 1919.96 mm, Length = 2441.89 mm


In [ ]:
results = yolo_model(image, conf=0.3)  # Reduce confidence threshold from default (0.5)



0: 640x480 4 Nails, 918.2ms
Speed: 7.9ms preprocess, 918.2ms inference, 24.2ms postprocess per image at shape (1, 3, 640, 480)


In [ ]:
def detect_objects(image):
    """Detect objects using YOLO model."""
    results = yolo_model(image)
    return results[0].boxes.xyxy.cpu().numpy()  # Bounding boxes

def estimate_depth(image):
    """Estimate depth using MiDaS."""
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    input_tensor = midas_transform(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        depth_map = midas(input_tensor)

    depth_map = depth_map.squeeze().cpu().numpy()
    depth_map = cv2.resize(depth_map, (image.shape[1], image.shape[0]))  # Resize back
    return depth_map

def get_real_world_size(bbox, depth_map, ref_bbox, ref_size_mm=85.6):
    """
    Compute real-world size using depth estimation and a known reference object.
    bbox: Bounding box (x1, y1, x2, y2)
    depth_map: Estimated depth map
    ref_bbox: Bounding box of reference object (e.g., credit card)
    ref_size_mm: Known width of reference object (default: 85.6 mm)
    """

    x1, y1, x2, y2 = map(int, bbox)
    obj_depth = np.median(depth_map[y1:y2, x1:x2])

    ref_x1, ref_y1, ref_x2, ref_y2 = map(int, ref_bbox)
    ref_depth = np.median(depth_map[ref_y1:ref_y2, ref_x1:ref_x2])

    ref_width_px = ref_x2 - ref_x1  # Reference object width in pixels
    pixel_to_mm = ref_size_mm / ref_width_px  # mm per pixel at reference depth

    # Adjust for depth ratio (relative scaling)
    scale_factor = obj_depth / ref_depth

    width_mm = (x2 - x1) * pixel_to_mm * scale_factor
    length_mm = (y2 - y1) * pixel_to_mm * scale_factor

    return round(width_mm, 2), round(length_mm, 2)

def main(image_path):
    """Run detection and measurement pipeline."""
    image = cv2.imread(image_path)

    # Detect objects
    boxes = detect_objects(image)
    if len(boxes) < 2:
        print("Error: No reference object detected!")
        return

    # Assume first detection is reference object
    ref_bbox = boxes[0]  # First object is the reference (credit card)
    nail_boxes = boxes[1:]  # Remaining are nails

    # Estimate depth
    depth_map = estimate_depth(image)

    print(f"Detections found: {len(nail_boxes)}")
    for i, bbox in enumerate(nail_boxes):
        width, length = get_real_world_size(bbox, depth_map, ref_bbox)
        print(f"Nail {i+1}: Width = {width} mm, Length = {length} mm")

        x1, y1, x2, y2 = map(int, bbox)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, f"{width}mm x {length}mm", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

# Run
image_path = "/content/ok.jfif"
main(image_path)


0: 640x480 4 Nails, 589.2ms
Speed: 4.4ms preprocess, 589.2ms inference, 14.0ms postprocess per image at shape (1, 3, 640, 480)
Detections found: 3
Nail 1: Width = 100.07 mm, Length = 113.46 mm
Nail 2: Width = 81.02 mm, Length = 92.49 mm
Nail 3: Width = 83.33 mm, Length = 106.18 mm


In [ ]:
# Run
image_path = "/content/oka.jfif"
main(image_path)


0: 640x480 4 Nails, 579.4ms
Speed: 4.9ms preprocess, 579.4ms inference, 15.6ms postprocess per image at shape (1, 3, 640, 480)
Detections found: 3
Nail 1: Width = 91.7 mm, Length = 101.73 mm
Nail 2: Width = 92.79 mm, Length = 106.29 mm
Nail 3: Width = 75.85 mm, Length = 96.47 mm
